[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Open-Medica/open-medical-skills/blob/main/oms-models/notebooks/05_evaluate_models.ipynb)

# 05 - Model Evaluation & Benchmarking

**OMS Custom Embedding Model Pipeline - Step 5 of 6**

This notebook benchmarks all embedding models on the OMS test set using standard information retrieval metrics.
It compares baselines (off-the-shelf models) against our fine-tuned models.

**Models evaluated:**
1. nomic-embed-text (baseline via API)
2. ToolRAG-T1 (Harvard/MIT baseline)
3. EmbeddingGemma-300M (base, before fine-tuning)
4. GTE-Qwen2-1.5B (base, before fine-tuning)
5. oms-toolrag-gemma-v1 (our fine-tuned EmbeddingGemma)
6. oms-toolrag-qwen-v1 (our fine-tuned GTE-Qwen2, if trained)

**Metrics:** MRR@5, Recall@1, Recall@5, Recall@10, NDCG@10

---

## 1. Setup & Dependencies

In [ ]:
!pip install -q sentence-transformers torch datasets pyyaml matplotlib numpy

In [ ]:
import json
import os
import sys
from collections import defaultdict
from pathlib import Path
from typing import Optional

import numpy as np
import torch
from datasets import Dataset, load_from_disk
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Load Test Data

Load the held-out test queries and ground truth tool assignments from Step 2.

In [ ]:
# Load test set
test_records = None

for data_path in [
    Path('data/processed/oms_training_pairs'),
    Path('../data/processed/oms_training_pairs'),
    Path('/content/drive/MyDrive/oms-models/data/processed/oms_training_pairs'),
]:
    if data_path.exists():
        ds = load_from_disk(str(data_path))
        if 'test' in ds:
            test_records = list(ds['test'])
            print(f"Loaded test set from: {data_path}")
        break

if test_records is None:
    for path in [Path('data/processed/test.jsonl'), Path('../data/processed/test.jsonl')]:
        if path.exists():
            test_records = [json.loads(line) for line in open(path) if line.strip()]
            print(f"Loaded test set from: {path}")
            break

assert test_records is not None, "No test data found. Run notebooks 01 and 02 first."

test_queries = [r['query'] for r in test_records]
test_tool_names = [r['tool_name'] for r in test_records]
test_categories = [r.get('category', 'unknown') for r in test_records]
test_query_types = [r.get('query_type', 'unknown') for r in test_records]

# Build ground truth mapping: query_idx -> set of relevant tool names
# (a query may match multiple tools if they have overlapping descriptions)
ground_truth = {i: {test_tool_names[i]} for i in range(len(test_queries))}

print(f"Test queries: {len(test_queries)}")
print(f"Unique tools in test: {len(set(test_tool_names))}")
print(f"Categories: {len(set(test_categories))}")
print(f"Query types: {len(set(test_query_types))}")

## 3. Define Evaluation Metrics

Implement standard IR metrics:
- **MRR@K** (Mean Reciprocal Rank): Average of 1/rank of the first relevant result
- **Recall@K**: Fraction of queries where the correct tool appears in top-K
- **NDCG@K** (Normalized Discounted Cumulative Gain): Position-aware relevance metric

In [ ]:
def compute_mrr_at_k(rankings: list[list[str]], ground_truth: dict[int, set[str]], k: int = 5) -> float:
    """Compute Mean Reciprocal Rank at K."""
    mrr_sum = 0.0
    for i, ranking in enumerate(rankings):
        relevant = ground_truth.get(i, set())
        for rank, item in enumerate(ranking[:k], 1):
            if item in relevant:
                mrr_sum += 1.0 / rank
                break
    return mrr_sum / len(rankings)


def compute_recall_at_k(rankings: list[list[str]], ground_truth: dict[int, set[str]], k: int = 5) -> float:
    """Compute Recall at K (fraction of queries with correct result in top-K)."""
    hits = 0
    for i, ranking in enumerate(rankings):
        relevant = ground_truth.get(i, set())
        top_k = set(ranking[:k])
        if top_k & relevant:
            hits += 1
    return hits / len(rankings)


def compute_ndcg_at_k(rankings: list[list[str]], ground_truth: dict[int, set[str]], k: int = 10) -> float:
    """Compute NDCG at K."""
    ndcg_sum = 0.0
    for i, ranking in enumerate(rankings):
        relevant = ground_truth.get(i, set())
        dcg = 0.0
        for rank, item in enumerate(ranking[:k], 1):
            if item in relevant:
                dcg += 1.0 / np.log2(rank + 1)
        # Ideal DCG (all relevant items at top)
        idcg = sum(1.0 / np.log2(r + 1) for r in range(1, min(len(relevant), k) + 1))
        if idcg > 0:
            ndcg_sum += dcg / idcg
    return ndcg_sum / len(rankings)


def evaluate_model(
    model_name: str,
    query_embeddings: np.ndarray,
    corpus_embeddings: np.ndarray,
    corpus_names: list[str],
    ground_truth: dict[int, set[str]],
) -> dict[str, float]:
    """Run full evaluation for a model's embeddings."""
    # Compute similarities
    sims = cos_sim(
        torch.tensor(query_embeddings),
        torch.tensor(corpus_embeddings),
    )

    # Get rankings
    rankings = []
    for i in range(len(query_embeddings)):
        sorted_indices = sims[i].argsort(descending=True).tolist()
        ranking = [corpus_names[idx] for idx in sorted_indices]
        rankings.append(ranking)

    # Compute metrics
    results = {
        'model': model_name,
        'MRR@5': compute_mrr_at_k(rankings, ground_truth, k=5),
        'Recall@1': compute_recall_at_k(rankings, ground_truth, k=1),
        'Recall@5': compute_recall_at_k(rankings, ground_truth, k=5),
        'Recall@10': compute_recall_at_k(rankings, ground_truth, k=10),
        'NDCG@10': compute_ndcg_at_k(rankings, ground_truth, k=10),
    }

    return results, rankings


print("Evaluation functions defined.")

## 4. Load Corpus

Build the full corpus of tool descriptions (all 2,049 tools) for retrieval evaluation.

In [ ]:
# Load all records to build the full corpus
corpus = {}

# From all dataset splits
for data_path in [
    Path('data/processed/oms_training_pairs'),
    Path('../data/processed/oms_training_pairs'),
]:
    if data_path.exists():
        full_ds = load_from_disk(str(data_path))
        for split_name in full_ds:
            for r in full_ds[split_name]:
                if r['tool_name'] not in corpus:
                    corpus[r['tool_name']] = r['tool_description']
        break

# Also load from JSONL if needed
for jsonl_dir in [Path('data/processed'), Path('../data/processed')]:
    for jsonl_file in jsonl_dir.glob('*.jsonl'):
        for line in open(jsonl_file):
            if line.strip():
                r = json.loads(line)
                if r.get('tool_name') and r['tool_name'] not in corpus:
                    corpus[r['tool_name']] = r.get('tool_description', '')

corpus_names = list(corpus.keys())
corpus_descs = list(corpus.values())

print(f"Corpus size: {len(corpus_names)} unique tools")
print(f"Test queries: {len(test_queries)}")

# Verify all test tools are in the corpus
missing = set(test_tool_names) - set(corpus_names)
if missing:
    print(f"WARNING: {len(missing)} test tools not found in corpus: {list(missing)[:5]}")
else:
    print("All test tools present in corpus.")

## 5-10. Evaluate All Models

Load and evaluate each model. Results are collected for the comparison table.

Each model is loaded, encodes the test queries and corpus, and is evaluated on the standard metrics.

In [ ]:
all_results = []
all_rankings = {}

def load_and_evaluate(
    name: str,
    model_path: str,
    query_prompt: Optional[str] = None,
    batch_size: int = 64,
    trust_remote_code: bool = False,
) -> Optional[dict]:
    """Load a model, encode test data, and compute metrics."""
    try:
        print(f"\n{'='*60}")
        print(f"Evaluating: {name}")
        print(f"{'='*60}")

        model = SentenceTransformer(model_path, trust_remote_code=trust_remote_code)
        print(f"  Loaded from: {model_path}")
        print(f"  Embedding dim: {model.get_sentence_embedding_dimension()}")

        # Encode queries
        encode_kwargs = {'convert_to_numpy': True, 'show_progress_bar': True, 'batch_size': batch_size}
        if query_prompt:
            encode_kwargs['prompt'] = query_prompt

        q_emb = model.encode(test_queries, **encode_kwargs)

        # Encode corpus (no prompt for documents)
        c_emb = model.encode(corpus_descs, convert_to_numpy=True, show_progress_bar=True, batch_size=batch_size)

        # Evaluate
        results, rankings = evaluate_model(name, q_emb, c_emb, corpus_names, ground_truth)
        all_results.append(results)
        all_rankings[name] = rankings

        # Print results
        for metric, value in results.items():
            if metric != 'model':
                print(f"  {metric}: {value:.4f}")

        # Free GPU memory
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        return results

    except Exception as e:
        print(f"  SKIPPED: {e}")
        return None


print("Evaluation helper defined. Running model evaluations...")

In [ ]:
# 5. Baseline: ToolRAG-T1 (Harvard/MIT)
for path in ['models/toolrag-t1', '../models/toolrag-t1', 'oms-models/models/toolrag-t1']:
    if Path(path).exists():
        load_and_evaluate(
            'ToolRAG-T1 (baseline)',
            path,
            query_prompt='Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery: ',
        )
        break
else:
    load_and_evaluate(
        'ToolRAG-T1 (baseline)',
        'yiminzhan/ToolRAG-T1',
        query_prompt='Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery: ',
    )

In [ ]:
# 6. Baseline: EmbeddingGemma-300M (base, before fine-tuning)
for path in ['models/embeddinggemma-300m', '../models/embeddinggemma-300m', 'oms-models/models/embeddinggemma-300m']:
    if Path(path).exists():
        load_and_evaluate('EmbeddingGemma-300M (base)', path, trust_remote_code=True)
        break
else:
    load_and_evaluate('EmbeddingGemma-300M (base)', 'google/embeddinggemma', trust_remote_code=True)

In [ ]:
# 7. Baseline: GTE-Qwen2-1.5B (base, before fine-tuning)
for path in ['models/gte-qwen2-1.5b', '../models/gte-qwen2-1.5b', 'oms-models/models/gte-qwen2-1.5b']:
    if Path(path).exists():
        load_and_evaluate(
            'GTE-Qwen2-1.5B (base)',
            path,
            query_prompt='Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery: ',
            batch_size=16,
            trust_remote_code=True,
        )
        break
else:
    load_and_evaluate(
        'GTE-Qwen2-1.5B (base)',
        'Alibaba-NLP/gte-Qwen2-1.5B-instruct',
        query_prompt='Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery: ',
        batch_size=16,
        trust_remote_code=True,
    )

In [ ]:
# 8. Our model: oms-toolrag-gemma-v1 (fine-tuned EmbeddingGemma)
for path in ['oms-toolrag-gemma-v1', '../oms-toolrag-gemma-v1', '/content/drive/MyDrive/oms-models/checkpoints/oms-toolrag-gemma-v1']:
    if Path(path).exists():
        load_and_evaluate('oms-toolrag-gemma-v1 (ours)', path, trust_remote_code=True)
        break
else:
    print("oms-toolrag-gemma-v1 not found. Run notebook 03 first.")

In [ ]:
# 9. Our model: oms-toolrag-qwen-v1 (fine-tuned GTE-Qwen2)
for path in ['oms-toolrag-qwen-v1', '../oms-toolrag-qwen-v1', '/content/drive/MyDrive/oms-models/checkpoints/oms-toolrag-qwen-v1']:
    if Path(path).exists():
        load_and_evaluate(
            'oms-toolrag-qwen-v1 (ours)',
            path,
            query_prompt='Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery: ',
            batch_size=16,
            trust_remote_code=True,
        )
        break
else:
    print("oms-toolrag-qwen-v1 not found. Run notebook 04 first (optional).")

## 11. Comparison Table

Side-by-side metrics for all evaluated models.

In [ ]:
if all_results:
    # Print markdown table
    metrics = ['MRR@5', 'Recall@1', 'Recall@5', 'Recall@10', 'NDCG@10']

    print(f"\n{'='*100}")
    print(f"MODEL COMPARISON")
    print(f"{'='*100}")

    # Header
    header = f"{'Model':<35}" + ''.join(f"{m:>12}" for m in metrics)
    print(header)
    print('-' * len(header))

    # Find best values per metric for highlighting
    best = {m: max(r[m] for r in all_results) for m in metrics}

    for r in all_results:
        row = f"{r['model']:<35}"
        for m in metrics:
            val = r[m]
            marker = ' *' if val == best[m] else '  '
            row += f"{val:>10.4f}{marker}"
        print(row)

    print(f"\n* = best in column")

    # Print as markdown (for reports)
    print(f"\n### Markdown Format:")
    print(f"\n| Model | " + ' | '.join(metrics) + ' |')
    print(f"| --- | " + ' | '.join(['---:'] * len(metrics)) + ' |')
    for r in all_results:
        vals = ' | '.join(f"{r[m]:.4f}" for m in metrics)
        print(f"| {r['model']} | {vals} |")
else:
    print("No models were evaluated. Check model paths.")

In [ ]:
# Bar chart comparison
if all_results:
    import matplotlib.pyplot as plt

    metrics = ['MRR@5', 'Recall@1', 'Recall@5', 'Recall@10', 'NDCG@10']
    model_names = [r['model'] for r in all_results]
    x = np.arange(len(metrics))
    width = 0.8 / len(model_names)

    fig, ax = plt.subplots(figsize=(14, 6))

    colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#F44336', '#00BCD4']

    for i, r in enumerate(all_results):
        values = [r[m] for m in metrics]
        offset = (i - len(model_names) / 2 + 0.5) * width
        bars = ax.bar(x + offset, values, width, label=r['model'],
                      color=colors[i % len(colors)], alpha=0.85)

    ax.set_xlabel('Metric')
    ax.set_ylabel('Score')
    ax.set_title('OMS Embedding Model Benchmark')
    ax.set_xticks(x)
    ax.set_xticklabels(metrics)
    ax.set_ylim(0, 1.0)
    ax.legend(loc='upper left', fontsize=8)
    ax.grid(True, axis='y', alpha=0.3)

    plt.tight_layout()
    plt.savefig('benchmark_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Chart saved to: benchmark_comparison.png")

## 12. Per-Category Breakdown

Analyze which medical categories improved the most from fine-tuning.

In [ ]:
if all_rankings and len(all_results) >= 2:
    # Group test queries by category
    category_indices = defaultdict(list)
    for i, cat in enumerate(test_categories):
        category_indices[cat].append(i)

    print(f"\n{'='*80}")
    print(f"PER-CATEGORY RECALL@5")
    print(f"{'='*80}")

    # Compare first baseline vs best fine-tuned model
    baseline_name = all_results[0]['model']
    best_name = max(all_results, key=lambda r: r['Recall@5'])['model']

    if baseline_name in all_rankings and best_name in all_rankings:
        baseline_rankings = all_rankings[baseline_name]
        best_rankings = all_rankings[best_name]

        print(f"\n{'Category':<35} {'Count':>6} {baseline_name[:20]:>22} {best_name[:20]:>22} {'Delta':>8}")
        print('-' * 95)

        for cat in sorted(category_indices.keys()):
            indices = category_indices[cat]
            if len(indices) < 3:
                continue

            cat_gt = {i: ground_truth[i] for i in indices}

            # Baseline recall@5
            b_hits = sum(1 for i in indices if ground_truth[i] & set(baseline_rankings[i][:5]))
            b_recall = b_hits / len(indices)

            # Best model recall@5
            f_hits = sum(1 for i in indices if ground_truth[i] & set(best_rankings[i][:5]))
            f_recall = f_hits / len(indices)

            delta = f_recall - b_recall
            arrow = '+' if delta > 0 else ''
            print(f"  {cat:<33} {len(indices):>6} {b_recall:>20.1%} {f_recall:>20.1%} {arrow}{delta:>6.1%}")
else:
    print("Need at least 2 evaluated models for category comparison.")

## 13. Per-Query-Type Breakdown

Analyze which query types (direct, clinical scenario, research, etc.) benefit most from fine-tuning.

In [ ]:
if all_rankings and len(all_results) >= 2:
    # Group by query type
    qt_indices = defaultdict(list)
    for i, qt in enumerate(test_query_types):
        qt_indices[qt].append(i)

    print(f"\n{'='*80}")
    print(f"PER-QUERY-TYPE RECALL@5")
    print(f"{'='*80}")

    baseline_name = all_results[0]['model']
    best_name = max(all_results, key=lambda r: r['Recall@5'])['model']

    if baseline_name in all_rankings and best_name in all_rankings:
        baseline_rankings = all_rankings[baseline_name]
        best_rankings = all_rankings[best_name]

        print(f"\n{'Query Type':<25} {'Count':>6} {baseline_name[:20]:>22} {best_name[:20]:>22} {'Delta':>8}")
        print('-' * 85)

        for qt in sorted(qt_indices.keys()):
            indices = qt_indices[qt]
            if len(indices) < 3:
                continue

            b_hits = sum(1 for i in indices if ground_truth[i] & set(baseline_rankings[i][:5]))
            b_recall = b_hits / len(indices)

            f_hits = sum(1 for i in indices if ground_truth[i] & set(best_rankings[i][:5]))
            f_recall = f_hits / len(indices)

            delta = f_recall - b_recall
            arrow = '+' if delta > 0 else ''
            print(f"  {qt:<23} {len(indices):>6} {b_recall:>20.1%} {f_recall:>20.1%} {arrow}{delta:>6.1%}")
else:
    print("Need at least 2 evaluated models for query type comparison.")

## 14. Error Analysis

Examine the worst retrievals from the best model to understand failure modes.

In [ ]:
if all_rankings:
    best_name = max(all_results, key=lambda r: r['Recall@5'])['model']
    best_rankings = all_rankings[best_name]

    # Find misses (queries where correct tool is NOT in top 10)
    misses = []
    for i, ranking in enumerate(best_rankings):
        gt_tools = ground_truth[i]
        top10 = set(ranking[:10])
        if not (top10 & gt_tools):
            # Find where the correct tool actually ranked
            gt_tool = list(gt_tools)[0]
            actual_rank = ranking.index(gt_tool) + 1 if gt_tool in ranking else -1
            misses.append({
                'query_idx': i,
                'query': test_queries[i],
                'gt_tool': gt_tool,
                'actual_rank': actual_rank,
                'top3': ranking[:3],
                'category': test_categories[i],
                'query_type': test_query_types[i],
            })

    print(f"\n{'='*80}")
    print(f"ERROR ANALYSIS ({best_name})")
    print(f"{'='*80}")
    print(f"Total misses (not in top-10): {len(misses)} / {len(test_queries)} ({100*len(misses)/len(test_queries):.1f}%)")

    # Sort by worst rank
    misses.sort(key=lambda m: m['actual_rank'] if m['actual_rank'] > 0 else 999999, reverse=True)

    print(f"\nTop 10 worst retrievals:")
    for i, m in enumerate(misses[:10], 1):
        rank_str = f"#{m['actual_rank']}" if m['actual_rank'] > 0 else "NOT FOUND"
        print(f"\n  {i}. [{m['category']}] [{m['query_type']}]")
        print(f"     Query: {m['query'][:80]}")
        print(f"     Expected: {m['gt_tool']} (actual rank: {rank_str})")
        print(f"     Got top-3: {', '.join(m['top3'])}")

    # Failure mode analysis
    miss_categories = defaultdict(int)
    miss_query_types = defaultdict(int)
    for m in misses:
        miss_categories[m['category']] += 1
        miss_query_types[m['query_type']] += 1

    print(f"\nFailure distribution by category:")
    for cat, count in sorted(miss_categories.items(), key=lambda x: -x[1])[:5]:
        print(f"  {cat}: {count} misses")

    print(f"\nFailure distribution by query type:")
    for qt, count in sorted(miss_query_types.items(), key=lambda x: -x[1])[:5]:
        print(f"  {qt}: {count} misses")
else:
    print("No rankings available for error analysis.")

## 15. Save Results

Save the benchmark results as JSON and a markdown report.

In [ ]:
RESULTS_DIR = Path('benchmarks/results')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Save as JSON
results_json = RESULTS_DIR / 'benchmark_results.json'
with open(results_json, 'w') as f:
    json.dump(all_results, f, indent=2)
print(f"JSON results: {results_json}")

# Save as markdown report
report_path = RESULTS_DIR / 'benchmark_report.md'
with open(report_path, 'w') as f:
    f.write('# OMS Embedding Model Benchmark Results\n\n')
    f.write(f'Test set: {len(test_queries)} queries, {len(corpus_names)} tools\n\n')

    # Table
    metrics = ['MRR@5', 'Recall@1', 'Recall@5', 'Recall@10', 'NDCG@10']
    f.write('| Model | ' + ' | '.join(metrics) + ' |\n')
    f.write('| --- | ' + ' | '.join(['---:'] * len(metrics)) + ' |\n')
    for r in all_results:
        vals = ' | '.join(f"{r[m]:.4f}" for m in metrics)
        f.write(f"| {r['model']} | {vals} |\n")

    f.write('\n## Key Findings\n\n')
    if len(all_results) >= 2:
        best = max(all_results, key=lambda r: r['Recall@5'])
        f.write(f"Best model: **{best['model']}** (Recall@5: {best['Recall@5']:.4f})\n")

print(f"Markdown report: {report_path}")
print(f"\nDone! All evaluation results saved.")

---

## Next Steps

Proceed to **06_deploy_to_qdrant.ipynb** to deploy the best model's embeddings to Qdrant for production use.

**Files produced by this notebook:**
- `benchmarks/results/benchmark_results.json` -- Full metrics as JSON
- `benchmarks/results/benchmark_report.md` -- Human-readable markdown report
- `benchmark_comparison.png` -- Bar chart visualization